In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
from tqdm.notebook import tqdm

In [ ]:
# Directories
CODE_DIR = Path(r"C:\Users\willi\.vscode\Github\ml-from-crowd")
DATA_DIR = Path(r"E:\Research_data\Stocktwits\dataset\v1\data\csv")
FIGURES_DIR = Path(r"C:\Users\willi\.vscode\Github\Figures")
MODEL_DATA_DIR = Path(r"C:\Users\willi\.vscode\Github\Data")

# Input files
INPUT_DATA = MODEL_DATA_DIR / "merged_master.pkl"

def find_all_features_file(model_type):
    """Resolve the 'all features' prediction filename for a model type by finding
    the largest input-count file on disk (excludes the 2-feature baseline), so this
    doesn't need updating whenever the feature set changes."""
    candidates = [
        p for p in MODEL_DATA_DIR.glob(f"predictions_{model_type}_input=*.pkl")
        if int(p.stem.split('input=')[1]) != 2
    ]
    if not candidates:
        return f"predictions_{model_type}_input=NOT_FOUND.pkl"
    return max(candidates, key=lambda p: int(p.stem.split('input=')[1])).name

# Model prediction files
# Key = column name in the dataframe, Value = prediction filename
MODELS = {
    'lr_2': 'predictions_linear_regression_input=2.pkl',
    'lr_all': find_all_features_file('linear_regression'),
}

# Time range to run regressions
START_DATE = '2012-01-01'
END_DATE = '2022-12-31'

# Prediction target
TARGET = 'f_cumret1'

# Put together all predictions

In [ ]:
# Load the original aggregated tweets data
df = pd.read_pickle(INPUT_DATA)[['date', 'permno', 'ticker', TARGET, 'log_volume']].copy()
df = df[(df['date'] >= START_DATE) & (df['date'] <= END_DATE)]

for model_col, filename in MODELS.items():
    # Load model predictions
    model_predictions = pd.read_pickle(MODEL_DATA_DIR / filename).drop(columns=['index', 'ticker'])
    model_predictions.columns = ['date', 'permno', model_col]
    
    # Merge with master data
    df = pd.merge(df, model_predictions, on=['date', 'permno'])  # Clean merge

In [ ]:
# Cross-sectionally de-mean the target and predictions each day
daily_means = df.groupby('date')[[TARGET] + list(MODELS.keys())].transform('mean')
df[TARGET] = df[TARGET] - daily_means[TARGET]
for col in list(MODELS.keys()):
    df[col] = df[col] - daily_means[col]

print("De-meaned target and predictions by date")
print(f"  Mean of {TARGET} after de-meaning: {df[TARGET].mean():.2e}")
for col in list(MODELS.keys()):
    print(f"  Mean of {col} after de-meaning: {df[col].mean():.2e}")

# Run sentiment regressions

In [ ]:
reg_results = []
for pred_col in tqdm(MODELS.keys(), desc="Running regressions"):
    reg_data = df[['permno','date', TARGET, pred_col]].dropna().copy()
    reg_data['date'] = pd.to_datetime(reg_data['date'])
    reg_data['date'] = reg_data['date'].dt.year * 10000 + reg_data['date'].dt.month*100 + reg_data['date'].dt.day 
    reg_data = reg_data.rename(columns={pred_col: 'pred'})
    y = reg_data[TARGET]
    # X = sm.add_constant(reg_data['pred'])
    X = reg_data['pred']
    model = sm.OLS(y, X, missing='drop').fit(cov_type='cluster',cov_kwds={'groups':np.array(reg_data[['permno','date']])})
    reg_results.append(model)

# Render the results in a table

In [ ]:
from latex_table import linear_regression

# Format and save the table
vars_to_include = ['pred', 'const']
var_names = ["Prediction", "Const."]
rename_dict = dict(zip(vars_to_include, var_names))

tbl = linear_regression(reg_results)
tbl.rename_variables(rename_dict)
tbl.columns = MODELS.keys()
tbl.obs = True
tbl.R2 = True
tbl.float_format = ".6f"
tbl.render(midrule=True)
tbl.tbl